> Adapted from *Dive into Deep Learning*, section [Implementation of Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/mlp-implementation.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 5.1: Multilayer Perceptrons](05.01_mlp.ipynb) | Next: [Section 5.3: Forward Propagation, Backward Propagation, and Computational Graphs](05.03_backprop.ipynb)

# Implementation of Multilayer Perceptrons

Multilayer perceptrons (MLPs) are not much more complex to implement than simple linear models. The key conceptual
difference is that we now concatenate multiple layers.

In [ ]:
import autograd.numpy as np
from divelet_ml import nn

import divelet_ml as d2l

## Implementation from Scratch

Let's begin again by implementing such a network from scratch.

### Initializing Model Parameters

Recall that Fashion-MNIST contains 10 classes,
and that each image consists of a $28 \times 28 = 784$
grid of grayscale pixel values.
As before we will disregard the spatial structure
among the pixels for now,
so we can think of this as a classification dataset
with 784 input features and 10 classes.
To begin, we will implement an MLP
with one hidden layer and 256 hidden units.
Both the number of layers and their width are adjustable
(they are considered hyperparameters).
Typically, we choose the layer widths to be divisible by larger powers of 2.
This is computationally efficient due to the way
memory is allocated and addressed in hardware.

Again, we will represent our parameters with several tensors.
Note that *for every layer*, we must keep track of
one weight matrix and one bias vector.
As always, we allocate memory
for the gradients of the loss with respect to these parameters.

In the code below, the trainer treats the arrays that are attributes of the model
as its parameters, to be tracked by `autograd` ([Section 2.5](https://d2l.ai/chapter_preliminaries/autograd.html#sec-autograd)).

In [ ]:
class MLPScratch(d2l.Classifier):
    def __init__(self, num_inputs, num_outputs, num_hiddens, lr, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        self.W1 = (np.random.randn(num_inputs, num_hiddens) * sigma).astype(np.float32)
        self.b1 = np.zeros(num_hiddens, dtype=np.float32)
        self.W2 = (np.random.randn(num_hiddens, num_outputs) * sigma).astype(np.float32)
        self.b2 = np.zeros(num_outputs, dtype=np.float32)

### Model

To make sure we know how everything works,
we will implement the ReLU activation ourselves
rather than invoking the built-in `relu` function directly.

In [ ]:
def relu(X):
    return np.maximum(X, 0)

Since we are disregarding spatial structure,
we `reshape` each two-dimensional image into
a flat vector of length  `num_inputs`.
Finally, we implement our model
with just a few lines of code. Since we use autograd this is all that it takes.

In [ ]:
@d2l.add_to_class(MLPScratch)
def forward(self, X):
    X = X.reshape((-1, self.num_inputs))
    H = relu(np.matmul(X, self.W1) + self.b1)
    return np.matmul(H, self.W2) + self.b2

### Training

Fortunately, the training loop for MLPs
is exactly the same as for softmax regression. We define the model, data, and trainer, then finally invoke the `fit` method on model and data.

In [ ]:
model = MLPScratch(num_inputs=784, num_outputs=10, num_hiddens=256, lr=0.1)
data = d2l.FashionMNIST(batch_size=256)
trainer = d2l.Trainer(max_epochs=d2l.browser_epochs(10))
trainer.fit(model, data)

## Concise Implementation

As you might expect, by relying on the high-level APIs, we can implement MLPs even more concisely.

### Model

Compared with our concise implementation
of softmax regression implementation
([Section 4.5](https://d2l.ai/chapter_linear-classification/softmax-regression-concise.html#sec-softmax-concise)),
the only difference is that we add
*two* fully connected layers where we previously added only *one*.
The first is the hidden layer,
the second is the output layer.

In [ ]:
class MLP(d2l.Classifier):
    def __init__(self, num_outputs, num_hiddens, lr):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(nn.Flatten(), nn.LazyLinear(num_hiddens),
                                 nn.ReLU(), nn.LazyLinear(num_outputs))

Previously, we defined `forward` methods for models to transform input using the model parameters.
These operations are essentially a pipeline:
you take an input and
apply a transformation (e.g.,
matrix multiplication with weights followed by bias addition),
then repetitively use the output of the current transformation as
input to the next transformation.
However, you may have noticed that 
no `forward` method is defined here.
In fact, `MLP` inherits the `forward` method from the `Module` class ([Section 3.2.2](https://d2l.ai/chapter_linear-regression/oo-design.html#subsec-oo-design-models)) to 
simply invoke `self.net(X)` (`X` is input),
which is now defined as a sequence of transformations
via the `Sequential` class.
The `Sequential` class abstracts the forward process
enabling us to focus on the transformations.
We will further discuss how the `Sequential` class works in [Section 6.1.2](https://d2l.ai/chapter_builders-guide/model-construction.html#subsec-model-construction-sequential).

### Training

The training loop is exactly the same
as when we implemented softmax regression.
This modularity enables us to separate
matters concerning the model architecture
from orthogonal considerations.

In [ ]:
model = MLP(num_outputs=10, num_hiddens=256, lr=0.1)
trainer.fit(model, data)

## Summary

Now that we have more practice in designing deep networks, the step from a single to multiple layers of deep networks does not pose such a significant challenge any longer. In particular, we can reuse the training algorithm and data loader. Note, though, that implementing MLPs from scratch is nonetheless messy: naming and keeping track of the model parameters makes it difficult to extend models. For instance, imagine wanting to insert another layer between layers 42 and 43. This might now be layer 42b, unless we are willing to perform sequential renaming. Moreover, if we implement the network from scratch, it is much more difficult for the framework to perform meaningful performance optimizations.

Nonetheless, you have now reached the state of the art of the late 1980s when fully connected deep networks were the method of choice for neural network modeling. Our next conceptual step will be to consider images. Before we do so, we need to review a number of statistical basics and details on how to compute models efficiently.

## Exercises

1. Change the number of hidden units `num_hiddens` and plot how its number affects the accuracy of the model. What is the best value of this hyperparameter?
1. Try adding a hidden layer to see how it affects the results.
1. Why is it a bad idea to insert a hidden layer with a single neuron? What could go wrong?
1. How does changing the learning rate alter your results? With all other parameters fixed, which learning rate gives you the best results? How does this relate to the number of epochs?
1. Let's optimize over all hyperparameters jointly, i.e., learning rate, number of epochs, number of hidden layers, and number of hidden units per layer.
    1. What is the best result you can get by optimizing over all of them?
    1. Why it is much more challenging to deal with multiple hyperparameters?
    1. Describe an efficient strategy for optimizing over multiple parameters jointly.
1. Compare the speed of the framework and the from-scratch implementation for a challenging problem. How does it change with the complexity of the network?
1. Measure the speed of tensor--matrix multiplications for well-aligned and misaligned matrices. For instance, test for matrices with dimension 1024, 1025, 1026, 1028, and 1032.
    1. How does this change between GPUs and CPUs?
    1. Determine the memory bus width of your CPU and GPU.
1. Try out different activation functions. Which one works best?
1. Is there a difference between weight initializations of the network? Does it matter?

Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/93)

Previous: [Section 5.1: Multilayer Perceptrons](05.01_mlp.ipynb) | Next: [Section 5.3: Forward Propagation, Backward Propagation, and Computational Graphs](05.03_backprop.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Implementation of Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/mlp-implementation.html)
(source: [`chapter_multilayer-perceptrons/mlp-implementation.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_multilayer-perceptrons/mlp-implementation.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Training for 10 epochs (10 passes over the 60,000 training images) takes minutes in the browser, so `d2l.browser_epochs(10)` gives 10 epochs outside JupyterLite but 3 in xeus-python and 1 in Pyodide, whose NumPy has no BLAS and multiplies matrices about 8 times slower. This is not in the book: in the browser, the curves show less training, and the accuracies are lower than the book's. Set `max_epochs=10` to train as long as the book does.
- NumPy has no automatic differentiation, so gradients come from [autograd](https://github.com/HIPS/autograd), as in [2.5](https://d2l.ai/chapter_preliminaries/autograd.html): `d2l.param_grads` differentiates the training step with respect to the model's parameter arrays, in place of PyTorch's `loss.backward()`, and the code imports autograd's NumPy (`import autograd.numpy as np`) so that the loss can be differentiated.
- NumPy draws random numbers in `float64`, so the model's parameters are cast with `.astype(np.float32)`: PyTorch's default is `float32`, the data is `float32`, and a `float64` product is about twice as slow, which matters in the browser.
- `nn.Parameter(...)` is dropped: the trainer finds the parameters among the model's arrays. `torch.max(X, a)` becomes `np.maximum(X, 0)`; `d2l.reshape` and `d2l.matmul` become `reshape` and `np.matmul`. `torch.nn` is divelet-ml's `nn` module.
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python). The word *tensor* is kept and means a NumPy `ndarray`.
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`. `%matplotlib inline` is removed: both JupyterLite kernels display plots inline.
- Equations keep the book's numbers. Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.